In [15]:
from pathlib import Path
import pandas as pd

# set path
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

project_root


WindowsPath('c:/Users/Dell - i5 11th Gen/Desktop/clinvar-gnomad-vus-analyzer')

In [16]:
# set candidates

candidate_path = (
    project_root
    / "data"
    / "processed"
    / "atm_candidates_with_variant_id.csv"
)

candidate_variants = pd.read_csv(candidate_path)
candidate_variants.head()

,Variation,Genes,Type,Condition,Classification,Review status,Molecular consequence,Protein change,Germline date last evaluated,Somatic clinical impact date last evaluated,...,GRCh37 Location,GRCh38 Location,Variation ID,VCV accession,rs_ID,chromosome,position,reference_allele,alternate_allele,variant_id
0,NM_000051.4(ATM):c.4A>C (p.Ser2Arg),ATM,single nucleotide variant,Hereditary cancer-predisposing syndrome,G: Uncertain significance,"criteria provided, single submitter",missense variant,S2R,2025/08/26,NaN,...,11:108098355,11:108227628,4170131,VCV004170131,rs4279232,11,108227628,A,C,11-108227628-A-C
1,NM_000051.4(ATM):c.4A>G (p.Ser2Gly),ATM,single nucleotide variant,"Hereditary cancer-predisposing syndrome, Famil...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,S2G,2024/02/19,NaN,...,11:108098355,11:108227628,640846,VCV000640846,rs639367,11,108227628,A,G,11-108227628-A-G
2,NM_000051.4(ATM):c.5G>C (p.Ser2Thr),ATM,single nucleotide variant,not specified,G: Uncertain significance,"criteria provided, single submitter",missense variant,S2T,2014/06/23,NaN,...,11:108098356,11:108227629,181943,VCV000181943,rs180377,11,108227629,G,C,11-108227629-G-C
3,NM_000051.4(ATM):c.6T>G (p.Ser2Arg),ATM,single nucleotide variant,not specified,G: Uncertain significance,"criteria provided, single submitter",missense variant,S2R,2019/12/09,NaN,...,11:108098357,11:108227630,1337452,VCV001337452,rs1328461,11,108227630,T,G,11-108227630-T-G
4,NM_000051.4(ATM):c.6T>A (p.Ser2Arg),ATM,single nucleotide variant,"Hereditary cancer-predisposing syndrome, Ataxi...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,S2R,2025/11/01,NaN,...,11:108098357,11:108227630,922075,VCV000922075,rs911284,11,108227630,T,A,11-108227630-T-A


In [17]:
# get gnomad_variants
import requests

gnomad_url = "https://gnomad.broadinstitute.org/api/"

query = """
{
    gene(gene_symbol:"ATM", reference_genome:GRCh38)
    {  
        variants(dataset: gnomad_r4)
        {
            variant_id
            joint{
                ac
                an
                homozygote_count
                filters
            }
        }
    }
}
"""

response = requests.post(
    gnomad_url,
    json ={"query": query},
    timeout=180
)

response.raise_for_status()

gnomad_response = response.json()

if "errors" in gnomad_response:
    raise ValueError(gnomad_response["errors"])
else:
    gnomad_variants = gnomad_response["data"]["gene"]["variants"]
    print("gnomad variants retrieved ", len(gnomad_variants))

gnomad variants retrieved  13905


In [18]:
# create gnomad_variants df
gnomad_df = pd.json_normalize(gnomad_variants)
gnomad_df.head()


,variant_id,joint.ac,joint.an,joint.homozygote_count,joint.filters
0,11-108227550-A-G,8,1165672,0,[]
1,11-108227551-T-C,505,1185976,3,[]
2,11-108227551-TATATAC-T,34,1185980,0,[]
3,11-108227552-A-C,1,1197114,0,[]
4,11-108227552-A-G,2,1196996,0,[]


In [19]:
# rename df cols
gnomad_population = (gnomad_df.rename(
    columns={
        "joint.ac": "gnomad_ac",
        "joint.an": "gnomad_an",
        "joint.homozygote_count": "gnomad_homozygote_count",
        "joint.filters": "gnomad_filters"
    }
)
.drop_duplicates(
    subset="variant_id"
)
.reset_index(drop=True)
)

gnomad_population.head()

,variant_id,gnomad_ac,gnomad_an,gnomad_homozygote_count,gnomad_filters
0,11-108227550-A-G,8,1165672,0,[]
1,11-108227551-T-C,505,1185976,3,[]
2,11-108227551-TATATAC-T,34,1185980,0,[]
3,11-108227552-A-C,1,1197114,0,[]
4,11-108227552-A-G,2,1196996,0,[]


In [20]:
# inner merge gnomad and candidate variants
matched_variants = candidate_variants.merge(
    gnomad_population, 
    on="variant_id",
    how="inner"
)
matched_variants.head()

,Variation,Genes,Type,Condition,Classification,Review status,Molecular consequence,Protein change,Germline date last evaluated,Somatic clinical impact date last evaluated,...,rs_ID,chromosome,position,reference_allele,alternate_allele,variant_id,gnomad_ac,gnomad_an,gnomad_homozygote_count,gnomad_filters
0,NM_000051.4(ATM):c.4A>G (p.Ser2Gly),ATM,single nucleotide variant,"Hereditary cancer-predisposing syndrome, Famil...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,S2G,2024/02/19,NaN,...,rs639367,11,108227628,A,G,11-108227628-A-G,2,1613596,0,[]
1,NM_000051.4(ATM):c.6T>G (p.Ser2Arg),ATM,single nucleotide variant,not specified,G: Uncertain significance,"criteria provided, single submitter",missense variant,S2R,2019/12/09,NaN,...,rs1328461,11,108227630,T,G,11-108227630-T-G,2,1613628,0,[]
2,NM_000051.4(ATM):c.10G>T (p.Val4Leu),ATM,single nucleotide variant,"Hereditary cancer-predisposing syndrome, Ataxi...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,V4L,2025/01/12,NaN,...,rs475312,11,108227634,G,T,11-108227634-G-T,2,1613478,0,[]
3,NM_000051.4(ATM):c.37C>T (p.Arg13Cys),ATM,single nucleotide variant,"not specified, Hereditary cancer-predisposing ...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,R13C,2025/12/20,NaN,...,rs183070,11,108227661,C,T,11-108227661-C-T,18,1613346,0,[]
4,NM_000051.4(ATM):c.38G>A (p.Arg13His),ATM,single nucleotide variant,"not specified, Hereditary cancer-predisposing ...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,R13H,2026/01/26,NaN,...,rs186132,11,108227662,G,A,11-108227662-G-A,17,1613566,0,[]


In [21]:
# convert type to numeric
matched_variants["gnomad_an"] = pd.to_numeric(matched_variants["gnomad_an"], errors="coerce")
matched_variants["gnomad_ac"] = pd.to_numeric(matched_variants["gnomad_ac"], errors="coerce")
matched_variants["gnomad_homozygote_count"] = pd.to_numeric(
    matched_variants["gnomad_homozygote_count"],
    errors="coerce"
)

In [22]:
# create af
matched_variants["gnomad_af"] = (
    matched_variants["gnomad_ac"] / matched_variants["gnomad_an"].replace(0, pd.NA)
    )

matched_variants.head()

,Variation,Genes,Type,Condition,Classification,Review status,Molecular consequence,Protein change,Germline date last evaluated,Somatic clinical impact date last evaluated,...,chromosome,position,reference_allele,alternate_allele,variant_id,gnomad_ac,gnomad_an,gnomad_homozygote_count,gnomad_filters,gnomad_af
0,NM_000051.4(ATM):c.4A>G (p.Ser2Gly),ATM,single nucleotide variant,"Hereditary cancer-predisposing syndrome, Famil...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,S2G,2024/02/19,NaN,...,11,108227628,A,G,11-108227628-A-G,2,1613596,0,[],0.000001
1,NM_000051.4(ATM):c.6T>G (p.Ser2Arg),ATM,single nucleotide variant,not specified,G: Uncertain significance,"criteria provided, single submitter",missense variant,S2R,2019/12/09,NaN,...,11,108227630,T,G,11-108227630-T-G,2,1613628,0,[],0.000001
2,NM_000051.4(ATM):c.10G>T (p.Val4Leu),ATM,single nucleotide variant,"Hereditary cancer-predisposing syndrome, Ataxi...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,V4L,2025/01/12,NaN,...,11,108227634,G,T,11-108227634-G-T,2,1613478,0,[],0.000001
3,NM_000051.4(ATM):c.37C>T (p.Arg13Cys),ATM,single nucleotide variant,"not specified, Hereditary cancer-predisposing ...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,R13C,2025/12/20,NaN,...,11,108227661,C,T,11-108227661-C-T,18,1613346,0,[],0.000011
4,NM_000051.4(ATM):c.38G>A (p.Arg13His),ATM,single nucleotide variant,"not specified, Hereditary cancer-predisposing ...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,R13H,2026/01/26,NaN,...,11,108227662,G,A,11-108227662-G-A,17,1613566,0,[],0.000011


In [23]:
# set for unmatched candidates
matched_variants_id = set(matched_variants["variant_id"])

unmatched_candidates = candidate_variants[
    ~candidate_variants["variant_id"].isin(matched_variants_id)
].copy()

unmatched_candidates.head()

,Variation,Genes,Type,Condition,Classification,Review status,Molecular consequence,Protein change,Germline date last evaluated,Somatic clinical impact date last evaluated,...,GRCh37 Location,GRCh38 Location,Variation ID,VCV accession,rs_ID,chromosome,position,reference_allele,alternate_allele,variant_id
0,NM_000051.4(ATM):c.4A>C (p.Ser2Arg),ATM,single nucleotide variant,Hereditary cancer-predisposing syndrome,G: Uncertain significance,"criteria provided, single submitter",missense variant,S2R,2025/08/26,NaN,...,11:108098355,11:108227628,4170131,VCV004170131,rs4279232,11,108227628,A,C,11-108227628-A-C
2,NM_000051.4(ATM):c.5G>C (p.Ser2Thr),ATM,single nucleotide variant,not specified,G: Uncertain significance,"criteria provided, single submitter",missense variant,S2T,2014/06/23,NaN,...,11:108098356,11:108227629,181943,VCV000181943,rs180377,11,108227629,G,C,11-108227629-G-C
4,NM_000051.4(ATM):c.6T>A (p.Ser2Arg),ATM,single nucleotide variant,"Hereditary cancer-predisposing syndrome, Ataxi...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,S2R,2025/11/01,NaN,...,11:108098357,11:108227630,922075,VCV000922075,rs911284,11,108227630,T,A,11-108227630-T-A
5,NM_000051.4(ATM):c.8T>C (p.Leu3Pro),ATM,single nucleotide variant,"Hereditary cancer-predisposing syndrome, Ataxi...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,L3P,2023/01/28,NaN,...,11:108098359,11:108227632,453759,VCV000453759,rs460775,11,108227632,T,C,11-108227632-T-C
7,NM_000051.4(ATM):c.13C>T (p.Leu5Phe),ATM,single nucleotide variant,Ataxia-telangiectasia syndrome,G: Uncertain significance,"criteria provided, single submitter",missense variant,L5F,2022/03/10,NaN,...,11:108098364,11:108227637,2108203,VCV002108203,rs2159617,11,108227637,C,T,11-108227637-C-T


In [24]:
assert matched_variants["variant_id"].notna().all()
assert matched_variants["variant_id"].is_unique


valid_counts = matched_variants[
    ["gnomad_ac", "gnomad_an"]
].dropna()

assert (valid_counts["gnomad_ac"] >= 0).all()
assert (valid_counts["gnomad_an"] > 0).all()

assert (
    valid_counts["gnomad_ac"]
    <= valid_counts["gnomad_an"]
).all()


assert (
    matched_variants["gnomad_homozygote_count"]
    .dropna()
    .ge(0)
    .all()
)

assert (
    matched_variants["gnomad_af"]
    .dropna()
    .between(0, 1)
    .all()
)


assert (
    len(matched_variants)
    + len(unmatched_candidates)
    == len(candidate_variants)
)


print("Validation checks passed!")

Validation checks passed!


In [25]:
match_percentage = (
    len(matched_variants)
    / len(candidate_variants)
    * 100
)

print("\nFinal pipeline summary")
print("----------------------")
print("ClinVar candidates:", len(candidate_variants))
print("Matched in gnomAD:", len(matched_variants))
print("Unmatched:", len(unmatched_candidates))
print(f"Match percentage: {match_percentage:.2f}%")


Final pipeline summary
----------------------
ClinVar candidates: 4534
Matched in gnomAD: 1474
Unmatched: 3060
Match percentage: 32.51%


In [26]:
# output unmatched candidates csv
output_file = (
    project_root
    / "data"
    / "processed"
    / "atm_clinvar_gnomad_unmatched.csv"
)

unmatched_candidates.to_csv(
    output_file,
    index=False
)

print("unmatched_candidates saved to: ", output_file)

unmatched_candidates saved to:  c:\Users\Dell - i5 11th Gen\Desktop\clinvar-gnomad-vus-analyzer\data\processed\atm_clinvar_gnomad_unmatched.csv


In [27]:
# output enriched variants csv
output_file = (
    project_root
    / "data"
    / "processed"
    / "atm_clinvar_gnomad_enriched.csv"
)

matched_variants.to_csv(
    output_file,
    index=False
)

print("matched_variants saved to:", output_file)

matched_variants saved to: c:\Users\Dell - i5 11th Gen\Desktop\clinvar-gnomad-vus-analyzer\data\processed\atm_clinvar_gnomad_enriched.csv
